In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install -q feedparser
!pip install -q datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 22.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-c

In [2]:
import pandas as pd
import numpy as np
import json
import feedparser
from tabulate import tabulate

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from datasets import Dataset

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

# Utils

In [3]:
def get_dataset(tokenizer, train_size=None, test_size=None):
    train_size = train_size or len(df_train)
    test_size = test_size or len(df_test)

    train_size = min(len(df_train), train_size)
    test_size = min(len(df_test), test_size)

    df_train_ = df_train.iloc[:train_size]
    df_test_ = df_test.iloc[:test_size]

    df_train_ = preprocess(df_train_, tokenizer)
    df_test_ = preprocess(df_test_, tokenizer)
    return df_train_, df_test_

In [4]:
def preprocess(df, tokenizer):
    df["prompt"] = get_message_prompts_tokenized(tokenizer, df["Title"].values, df["Summary"].values)
    df["labels"] = df["Category Description"]
    return df

In [5]:
def extract_label_from_output(generated_sentences, tokenizer):
    return [x.split("\n\nLabeled Category:")[-1].split(tokenizer.eos_token)[0].strip() for x in generated_sentences]

In [6]:
def get_message_prompts_tokenized(tokenizer, titles: list[str], abstracts: list[str]):
    messages = [
        [
            SYSTEM_PROMPT,
            {"role": "user", "content": f"Title: {title} \n Abstract: {abstract[:ABSTRACT_CUTOFF_LENGTH]} ..."},
            POST_MESSAGE
        ]
        for title, abstract in zip(titles, abstracts)

    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, continue_final_message=True
    )

In [7]:
def generate_input_output_pair(prompt, target_responses):
    chat_templates = tokenizer.apply_chat_template(prompt, continue_final_message=True, tokenize=False)
    full_response_text = [
        (chat_template + " " + target_response + tokenizer.eos_token)
        for chat_template, target_response in zip(chat_templates, target_responses)
    ]
    input_ids_tokenized = tokenizer(full_response_text, return_tensors="pt", add_special_tokens=False)["input_ids"]


    labels_tokenized = tokenizer([" " + response + tokenizer.eos_token for response in target_responses],
            add_special_tokens=False, return_tensors="pt", padding="max_length", max_length=input_ids_tokenized.shape[1])["input_ids"]

    labels_tokenized_fixed = torch.where(labels_tokenized != tokenizer.pad_token_id, labels_tokenized, -100)
    labels_tokenized_fixed[:, -1] = tokenizer.pad_token_id

    input_ids_tokenized_left_shifted = input_ids_tokenized[:, :-1]
    labels_tokenized_right_shifted = labels_tokenized_fixed[:, 1:]

    attention_mask = input_ids_tokenized_left_shifted != tokenizer.pad_token_id
    return {
        "input_ids": input_ids_tokenized_left_shifted,
        "attention_mask": attention_mask,
        "labels": labels_tokenized_right_shifted
    }

In [8]:
def calculate_loss(logits, labels):
    loss_fn = nn.CrossEntropyLoss(reduction='none')
    cross_entropy_loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
    return cross_entropy_loss

In [9]:
def generate_outputs(prompts: list[str], model, tokenizer):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenized = tokenizer(prompts, padding=True, return_tensors="pt", add_special_tokens=False).to(device)
    output_batch = model.generate(input_ids=tokenized["input_ids"], attention_mask=tokenized["attention_mask"],
                                      max_new_tokens=20, do_sample=False, top_p=None)
    decoded_batch = tokenizer.batch_decode(output_batch, skip_special_tokens=True)
    predictions = extract_label_from_output(decoded_batch, tokenizer)
    return predictions

In [10]:
def generate_output_from_input(model, tokenizer, title, abstract):
    prompts = get_message_prompts_tokenized(tokenizer, [title], [abstract])
    predictions = generate_outputs(prompts, model, tokenizer)
    return predictions

In [11]:
def print_colored_df(df):
    table = []
    for index, row in df.iterrows():
        pred, label = row['predictions'], row['labels']
        # Format each row as a list
        formatted_row = [index, pred, label]

        # Apply color based on the condition
        if pred == label:
            # Green for matching values
            colored_row = [f"\033[32m{item}\033[0m" for item in formatted_row]
        else:
            # Red for mismatching values
            colored_row = [f"\033[31m{item}\033[0m" for item in formatted_row]

        # Append the colored row to the table
        table.append(colored_row)

    # Print the table with headers
    print(tabulate(table, headers=['Index', 'Predictions', 'Labels'], tablefmt='grid'))

In [12]:
def test_model(dataloader, model, tokenizer):
    comparison_df = {
        "predictions": [],
        "labels": []
    }
    for batch in dataloader:
        predictions = generate_outputs(prompts=batch["prompt"], model=model, tokenizer=tokenizer)
        comparison_df["labels"].extend(batch["labels"])
        comparison_df["predictions"].extend(predictions)

    comparison_df = pd.DataFrame(comparison_df)
    accuracy = (comparison_df["labels"] == comparison_df["predictions"]).mean()
    num_invalid_pred = (~comparison_df["predictions"].isin(VALID_CLASSES)).mean()

    print_colored_df(comparison_df.head(10))
    return {"accuracy" : accuracy, "invalid prediction": num_invalid_pred}

# Dataset

In [17]:
random_seed = 42

ABSTRACT_CUTOFF_LENGTH = 150
csv_filename = "arxiv_dataset.csv"
VALID_CLASSES = [
    'Machine Learning',
    'Computer Vision and Pattern Recognition',
    'Computation and Language (Natural Language Processing)',
    'Robotics',
    'Cryptography and Security',
    'Artificial Intelligence'
]

df = pd.read_csv('/content/drive/MyDrive/arxiv_dataset.csv')
df = df[df["Category Description"].isin(VALID_CLASSES)]


# Shuffle the DataFrame
df = df.sample(frac=1, random_state=random_seed).reset_index(drop=True).reset_index()
train_size = 0.8
train_len = int(train_size * len(df))

df_train = df[:train_len]
df_test = df[train_len:]

In [18]:
print(df_train.shape)
df_train.head()

(1036, 6)


,index,Title,Category,Category Description,Published,Summary
0,0,Can LLMs Understand Time Series Anomalies?,cs.LG,Machine Learning,2024-10-07,Large Language Models (LLMs) have gained popul...
1,1,$\textit{X}^2$-DFD: A framework for e${X}$plai...,cs.CV,Computer Vision and Pattern Recognition,2024-10-08,Detecting deepfakes has become an important ta...
2,2,GRU-D Characterizes Age-Specific Temporal Miss...,cs.LG,Machine Learning,2024-10-07,"Temporal missingness, defined as unobserved pa..."
3,3,Can the Variation of Model Weights be used as ...,cs.CL,Computation and Language (Natural Language Pro...,2024-10-05,Many-to-one neural machine translation systems...
4,4,Output Scouting: Auditing Large Language Model...,cs.CL,Computation and Language (Natural Language Pro...,2024-10-04,Recent high profile incidents in which the use...


In [19]:
print(df_test.shape)
df_test.head()

(260, 6)


,index,Title,Category,Category Description,Published,Summary
1036,1036,SIA-OVD: Shape-Invariant Adapter for Bridging ...,cs.CV,Computer Vision and Pattern Recognition,2024-10-08,Open-vocabulary detection (OVD) aims to detect...
1037,1037,Polynomial Time Cryptanalytic Extraction of De...,cs.CR,Cryptography and Security,2024-10-08,Deep neural networks (DNNs) are valuable asset...
1038,1038,Task-oriented Time Series Imputation Evaluatio...,cs.LG,Machine Learning,2024-10-09,Time series analysis is widely used in many fi...
1039,1039,Towards Interpreting Visual Information Proces...,cs.CV,Computer Vision and Pattern Recognition,2024-10-09,Vision-Language Models (VLMs) are powerful too...
1040,1040,Submodular Optimization for Keyframe Selection...,cs.RO,Robotics,2024-10-08,Keyframes are LiDAR scans saved for future ref...


In [22]:
SYSTEM_PROMPT = \
    {
        "role": "system",
        "content": f"""You are an AI system that reads the title and abstract of a paper and classifies which area of computer science the paper belongs to.
No explanation required, you must choose from the following classes:
{" or ".join(["Labeled Category: " + x for x in list(VALID_CLASSES)])}.
Ensure your output is from the above list only.
        """,
    }


POST_MESSAGE = \
    {
        "role": "assistant",
        "content": "Labeled Category:"
    }

# Load Model

In [20]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"
device = torch.device("cuda")

tokenizer = AutoTokenizer.from_pretrained(model_id,
                                          padding_side="left",
                                          use_fast = True
                                          token = '[REDACTED]')
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id,
                                            torch_dtype=torch.bfloat16,
                                            token = '[REDACTED]',
                                            device_map=device)

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [13]:
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
if model.config.pad_token_id is None:
    model.config.pad_token_id = model.config.eos_token_id

In [27]:
num_test_examples = 15
batch_size = 16
device = torch.device("cuda") # change to your device

_, test_df = get_dataset(tokenizer=tokenizer, train_size=0, test_size=num_test_examples)
test_dataset = Dataset.from_pandas(test_df)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
metrics = test_model(test_dataloader, model, tokenizer)
print("Number of test examples: ", len(test_df))
print("\n".join([f"{k} = {v}" for k, v in metrics.items()]))

<ipython-input-8-0302bc8e3f20>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["prompt"] = get_message_prompts_tokenized(tokenizer, df["Title"].values, df["Summary"].values)
<ipython-input-8-0302bc8e3f20>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["labels"] = df["Category Description"]
<ipython-input-8-0302bc8e3f20>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docume

+---------+--------------------------------------------------------+-----------------------------------------+
|   Index | Predictions                                            | Labels                                  |
+=========+========================================================+=========================================+
|       0 | Computer Vision and Pattern Recognition                | Computer Vision and Pattern Recognition |
+---------+--------------------------------------------------------+-----------------------------------------+
|       1 | Cryptography and Security                              | Cryptography and Security               |
+---------+--------------------------------------------------------+-----------------------------------------+
|       2 | Machine Learning                                       | Machine Learning                        |
+---------+--------------------------------------------------------+-----------------------------------------+
|

# Tokenizers

In [28]:
device = torch.device("cuda")

input_prompt = [
    "Hello how are you doing tell me?",
    "The capital of India is"
]

tokenized = tokenizer(input_prompt, padding=True, return_tensors="pt").to(device)

print(tokenized["input_ids"].shape)

torch.Size([2, 9])


In [29]:
tokenized["input_ids"]

tensor([[128000,   9906,   1268,    527,    499,   3815,   3371,    757,     30],
        [128009, 128009, 128009, 128000,    791,   6864,    315,   6890,    374]],
       device='cuda:0')

In [30]:
tokenizer.batch_decode(tokenized["input_ids"])

['<|begin_of_text|>Hello how are you doing tell me?',
 '<|eot_id|><|eot_id|><|eot_id|><|begin_of_text|>The capital of India is']

In [31]:
tokenized["attention_mask"]

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 1, 1]], device='cuda:0')

In [32]:
VALID_CLASSES = ['Machine Learning',
                'Computer Vision and Pattern Recognition',
                'Computation and Language (Natural Language Processing)',
                'Robotics',
                'Cryptography and Security',
                'Artificial Intelligence']
SYSTEM_PROMPT = \
    {
        "role": "system",
        "content": f"""You are an AI system that reads the title and abstract of a paper and classifies which area of computer science the paper belongs to.
No explanation required, you must choose from the following classes:
{" or ".join(["Labeled Category: " + x for x in list(VALID_CLASSES)])}.
Ensure your output is from the above list only.
        """,
    }

POST_MESSAGE = \
    {
        "role": "assistant",
        "content": "Labeled Category:"
    }


USER_MESSAGES = [
{
    "role": "user",
    "content": f"Title: {row['Title']}\nAbstract: {row['Summary'][:100]}"
}
for _, row in df.iloc[:5].iterrows()
]

PROMPTS = [
    [SYSTEM_PROMPT, USER_MSG, POST_MESSAGE]
    for USER_MSG in USER_MESSAGES
]

In [33]:
print(SYSTEM_PROMPT["content"])

You are an AI system that reads the title and abstract of a paper and classifies which area of computer science the paper belongs to.
No explanation required, you must choose from the following classes:
Labeled Category: Machine Learning or Labeled Category: Computer Vision and Pattern Recognition or Labeled Category: Computation and Language (Natural Language Processing) or Labeled Category: Robotics or Labeled Category: Cryptography and Security or Labeled Category: Artificial Intelligence.
Ensure your output is from the above list only.
        


In [34]:
print(USER_MESSAGES[0]["content"])

Title: Can LLMs Understand Time Series Anomalies?
Abstract: Large Language Models (LLMs) have gained popularity in time series
forecasting, but their potential 


In [35]:
print(json.dumps(PROMPTS[0], indent=4))

[
    {
        "role": "system",
        "content": "You are an AI system that reads the title and abstract of a paper and classifies which area of computer science the paper belongs to.\nNo explanation required, you must choose from the following classes:\nLabeled Category: Machine Learning or Labeled Category: Computer Vision and Pattern Recognition or Labeled Category: Computation and Language (Natural Language Processing) or Labeled Category: Robotics or Labeled Category: Cryptography and Security or Labeled Category: Artificial Intelligence.\nEnsure your output is from the above list only.\n        "
    },
    {
        "role": "user",
        "content": "Title: Can LLMs Understand Time Series Anomalies?\nAbstract: Large Language Models (LLMs) have gained popularity in time series\nforecasting, but their potential "
    },
    {
        "role": "assistant",
        "content": "Labeled Category:"
    }
]


In [36]:
tokenized = tokenizer.apply_chat_template(
    PROMPTS,
    continue_final_message=True,
    padding=True, return_tensors="pt"
).to(device)

out = model.generate(tokenized, max_new_tokens=20)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [37]:
decoded = tokenizer.batch_decode(out)
decoded

['<|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are an AI system that reads the title and abstract of a paper and classifies which area of computer science the paper belongs to.\nNo explanation required, you must choose from the following classes:\nLabeled Category: Machine Learning or Labeled Category: Computer Vision and Pattern Recognition or Labeled Category: Computation and Language (Natural Language Processing) or Labeled Category: Robotics or Labeled Category: Cryptography and Security or Labeled Category: Artificial Intelligence.\nEnsure your output is from the above list only.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTitle: Can LLMs Understand Time Serie

In [38]:
def extract_label(d):
    try:
        return d.split("<|start_header_id|>assistant<|end_header_id|>\n\nLabeled Category:")[1] \
                .split("<|eot_id|>")[0].strip().replace("\xa0", " ")
    except IndexError:
        return None

labels = [extract_label(d) for d in decoded]

In [39]:
for i in labels:
    print("***")
    print(i)

***
Machine Learning
***
Machine Learning
***
Machine Learning
***
Computation and Language (Natural Language Processing)
***
Computation and Language (Natural Language Processing)


# Chat Templates

In [40]:
prompt = [
    {
        "role": "system",
        "content": "You are a smart AI assistant who speaks like a pirate."
    },
    {
        "role": "user",
        "content": "Where does the sun rise?"
    }

]
tokenizer.pad_token = tokenizer.eos_token

tokenized = tokenizer.apply_chat_template(
    prompt,
    add_generation_prompt=True,
    tokenize=True,
    padding=True,
    return_tensors="pt"
)# .to(device)

print(tokenized)

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,  10263,    220,   2366,     19,    271,   2675,    527,
            264,   7941,  15592,  18328,    889,  21881,   1093,    264,  55066,
             13, 128009, 128006,    882, 128007,    271,   9241,   1587,    279,
           7160,  10205,     30, 128009, 128006,  78191, 128007,    271]])


In [41]:
out = model.generate(tokenized.to(device), max_new_tokens=20)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [42]:
out

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,  10263,    220,   2366,     19,    271,   2675,    527,
            264,   7941,  15592,  18328,    889,  21881,   1093,    264,  55066,
             13, 128009, 128006,    882, 128007,    271,   9241,   1587,    279,
           7160,  10205,     30, 128009, 128006,  78191, 128007,    271,  87575,
            387,   1390,    258,      6,    311,   1440,   1405,    279,   7160,
          38268,     11,  36346,     30,    220,   8489,     11,  30276,     88,
             11]], device='cuda:0')

In [43]:
decoded = tokenizer.batch_decode(out)
print(decoded[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a smart AI assistant who speaks like a pirate.<|eot_id|><|start_header_id|>user<|end_header_id|>

Where does the sun rise?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Ye be wantin' to know where the sun rises, eh?  Well, matey,


# Continue Final Message

In [44]:
prompt_template = [
    {
        "role": "system",
        "content": "You are a smart AI assistant who speaks like a pirate."
    },
    {
        "role": "user",
        "content": "Where does the sun rises?"
    },
    {
        "role": "assistant",
        "content": "Aye aye"
    },

]
tokenizer.pad_token = tokenizer.eos_token

tokenized = tokenizer.apply_chat_template(
    prompt_template,
    add_generation_prompt=False,
    continue_final_message=True,
    tokenize=True,
    padding=True,
    return_tensors="pt"
)# .to(device)

print(tokenized)

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,  10263,    220,   2366,     19,    271,   2675,    527,
            264,   7941,  15592,  18328,    889,  21881,   1093,    264,  55066,
             13, 128009, 128006,    882, 128007,    271,   9241,   1587,    279,
           7160,  38268,     30, 128009, 128006,  78191, 128007,    271,     32,
           9188,    264,   9188]])


In [45]:
out = model.generate(tokenized.to(device), max_new_tokens=56)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [46]:
decoded = tokenizer.batch_decode(out)
print(decoded[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a smart AI assistant who speaks like a pirate.<|eot_id|><|start_header_id|>user<|end_header_id|>

Where does the sun rises?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Aye aye, matey! Ye be askin' where the sun rises, eh?  Well, listen close and I'll tell ye. The sun rises in the east, where the mornin' dew be glistenin' like a chest overflowin' with gold doublo


# Training

In [47]:
text = "Hello how are you today"
input_ids = tokenizer([text], return_tensors="pt")["input_ids"].to(device)
out = model(input_ids = input_ids)

In [48]:
tokenizer.decode(out.logits.argmax(axis=-1)[0, -1])

'?'

# Training on Sequence

In [49]:
sentence = ["Subscribe to Neural Breakdown with AVB"]
tokenized = tokenizer(sentence, return_tensors="pt")["input_ids"]
print(tokenized)
print(tokenizer.batch_decode(tokenized))

tensor([[128000,  29673,    311,  61577,  15996,   2996,    449,  12431,     33]])
['<|begin_of_text|>Subscribe to Neural Breakdown with AVB']


In [50]:
input_ids = tokenized[:, :-1] # (start) to the (end - 1)
target_ids = tokenized[:, 1:] # (start + 1) to (end)

print("Input Seq: ", input_ids)
print("Target Seq: ", target_ids)

Input Seq:  tensor([[128000,  29673,    311,  61577,  15996,   2996,    449,  12431]])
Target Seq:  tensor([[29673,   311, 61577, 15996,  2996,   449, 12431,    33]])


In [51]:
question = "Capital of India?"
answer = "New Delhi"

prompt = [
    {"role": "user", "content": "Capital of India?"},
    {"role": "assistant", "content": "Capital:"}
]
answer = "New Delhi"

chat_template = tokenizer.apply_chat_template(prompt, continue_final_message=True, tokenize=False)
print(chat_template)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Capital of India?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Capital:


In [52]:
full_response_text = chat_template + " " + answer + tokenizer.eos_token
print(full_response_text)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Capital of India?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Capital: New Delhi<|eot_id|>


In [53]:
tokenized = tokenizer(full_response_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
print(tokenized)

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,  10263,    220,   2366,     19,    271, 128009, 128006,
            882, 128007,    271,  64693,    315,   6890,     30, 128009, 128006,
          78191, 128007,    271,  64693,     25,   1561,  22767, 128009]])


In [54]:
input_ids = tokenized[:, :-1] # (start) to the (end - 1)
target_ids = tokenized[:, 1:] # (start + 1) to (end)

print(input_ids)
print(target_ids)

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,  10263,    220,   2366,     19,    271, 128009, 128006,
            882, 128007,    271,  64693,    315,   6890,     30, 128009, 128006,
          78191, 128007,    271,  64693,     25,   1561,  22767]])
tensor([[128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,     25,
           6790,    220,   2366,     18,    198,  15724,   2696,     25,    220,
           1627,  10263,    220,   2366,     19,    271, 128009, 128006,    882,
         128007,    271,  64693,    315,   6890,     30, 128009, 128006,  78191,
         128007,    271,  64693,     25,   1561,  22767, 128009]])


In [55]:
input_ids.shape

torch.Size([1, 43])

In [56]:
print(tokenizer.convert_ids_to_tokens(3648))
print(tokenizer.convert_ids_to_tokens(1561))

New
ĠNew


In [57]:
labels_tokenized = tokenizer([" " + answer], add_special_tokens=False, return_tensors="pt")["input_ids"]
print(labels_tokenized)

tensor([[ 1561, 22767]])


In [58]:
labels_tokenized = tokenizer([" " + answer + tokenizer.eos_token],
                             add_special_tokens=False, return_tensors="pt", padding="max_length", max_length=target_ids.shape[1])["input_ids"]

In [59]:
labels_tokenized

tensor([[128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009, 128009,
         128009, 128009, 128009, 128009,   1561,  22767, 128009]])

In [60]:
labels_tokenized_fixed = torch.where(labels_tokenized != tokenizer.pad_token_id, labels_tokenized, -100)
labels_tokenized_fixed

tensor([[ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          1561, 22767,  -100]])

In [61]:
labels_tokenized_fixed[:, -1] = tokenizer.eos_token_id
labels_tokenized_fixed

tensor([[  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   1561,  22767, 128009]])

In [62]:
tokenizer.convert_ids_to_tokens(22767)

'ĠDelhi'

In [63]:
tokenizer.convert_ids_to_tokens(3648)

'New'

In [64]:
data = generate_input_output_pair(
    prompt = [
        [
            {"role": "user", "content": "Capital of India?"},
            {"role": "assistant", "content": "Capital:"}
        ]
    ],
    target_responses=["New Delhi"]
)

In [65]:
data["input_ids"]

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,  10263,    220,   2366,     19,    271, 128009, 128006,
            882, 128007,    271,  64693,    315,   6890,     30, 128009, 128006,
          78191, 128007,    271,  64693,     25,   1561,  22767]])

In [66]:
data["labels"]

tensor([[  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   1561,  22767, 128009]])

# Loss

In [67]:
out = model(input_ids=data["input_ids"].to(device))

In [68]:
out.logits.shape

torch.Size([1, 43, 128256])

In [69]:
data["labels"].shape

torch.Size([1, 43])

In [70]:
calculate_loss(out.logits, data["labels"].to(device))

tensor([0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0059e-01, 5.0354e-04,
        7.8125e-01], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<NllLossBackward0>)

In [71]:
data = generate_input_output_pair(
    prompt = [
        [
            {"role": "user", "content": "Capital of India?"},
            {"role": "assistant", "content": "Capital:"}
        ]
    ],
    target_responses=["Mumbai"]
)

out = model(input_ids=data["input_ids"].to(device))

In [72]:
calculate_loss(out.logits, data["labels"].to(device))

tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
        12.0000,  3.4062], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<NllLossBackward0>)

In [73]:
out.logits.argmax(-1)

tensor([[ 14924, 128006,    198, 128006,    567,   1303,  48448,   5825,     25,
           5587,    220,   2366,     17,    271,  27504,    596,     25,    220,
           2437,   3799,    220,   2366,     18,    271,    791, 128006,  78191,
         128007,    271,    791,    315,    279,    374,    720, 128006,  78191,
         128007,    271,    791,    315,   1561,    374]], device='cuda:0')

In [ ]:
data["labels"]

tensor([[  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100, 128009]])

# Training

In [21]:
training_prompt = [
    {
        "role": "user", "content": "Who to subscribe to on YT for ML?"
    },
    {
        "role": "assistant", "content": "Subscribe to"
    }
]
target_response = "Neural Breakdown with AVB"

In [22]:
test_tokenized = tokenizer.apply_chat_template(training_prompt, continue_final_message=True, return_tensors="pt").to(device)
test_out = model.generate(test_tokenized, max_new_tokens=10)
print(tokenizer.batch_decode(test_out, skip_special_tokens=True)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 03 Apr 2025

user

Who to subscribe to on YT for ML?assistant

Subscribe to the following YouTube channels for Machine Learning (ML)


In [23]:
data = generate_input_output_pair(prompt=[training_prompt], target_responses=[target_response])
data["input_ids"] = data["input_ids"].to(device)
data["labels"] = data["labels"].to(device)

In [24]:
data["labels"][:, -7:]

tensor([[ 61577,  15996,   2996,    449,  12431,     33, 128009]],
       device='cuda:0')

In [25]:
tokenizer.batch_decode(data["labels"][:, -7:])

[' Neural Breakdown with AVB<|eot_id|>']

In [26]:
data["labels"]

tensor([[  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          61577,  15996,   2996,    449,  12431,     33, 128009]],
       device='cuda:0')

In [27]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [28]:
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

for _ in range(10):
    out = model(input_ids=data["input_ids"].to(device))
    loss = calculate_loss(out.logits, data["labels"]).mean()

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print("loss: ", loss.item())

loss:  1.1640625
loss:  0.359375
loss:  0.5
loss:  0.71484375
loss:  0.76171875
loss:  0.85546875
loss:  0.1923828125
loss:  0.0167236328125
loss:  0.0089111328125
loss:  0.0030059814453125


In [29]:
test_tokenized = tokenizer.apply_chat_template(training_prompt, continue_final_message=True, return_tensors="pt").to(device)
print(test_tokenized)
# model.generate(test_tokenized)

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   2839,   5186,    220,   2366,     20,    271, 128009, 128006,
            882, 128007,    271,  15546,    311,  18447,    311,    389,  83092,
            369,  20187,     30, 128009, 128006,  78191, 128007,    271,  29673,
            311]], device='cuda:0')


In [30]:
data["input_ids"]

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   2839,   5186,    220,   2366,     20,    271, 128009, 128006,
            882, 128007,    271,  15546,    311,  18447,    311,    389,  83092,
            369,  20187,     30, 128009, 128006,  78191, 128007,    271,  29673,
            311,  61577,  15996,   2996,    449,  12431,     33]],
       device='cuda:0')

In [31]:
test_out = model.generate(test_tokenized, max_new_tokens=10)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [32]:
print(tokenizer.batch_decode(test_out, skip_special_tokens=True)[0])

system

Cutting Knowledge Date: December 2023
Today Date: 03 Apr 2025

user

Who to subscribe to on YT for ML?assistant

Subscribe to Neural Breakdown with AVB


In [33]:
test_tokenized = tokenizer.apply_chat_template(training_prompt, continue_final_message=True, return_tensors="pt").to(device)
test_out = model.generate(test_tokenized, max_new_tokens=10)
print(tokenizer.batch_decode(test_out, skip_special_tokens=True)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 03 Apr 2025

user

Who to subscribe to on YT for ML?assistant

Subscribe to Neural Breakdown with AVB


# LoRA

In [15]:
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=32,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q_proj', 'v_proj']
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


In [16]:
training_prompt = [
    {
        "role": "user", "content": "Who to subscribe to on YT for ML?"
    },
    {
        "role": "assistant", "content": "Subscribe to"
    }
]
target_response = "Neural Breakdown with AVB"

In [17]:
test_tokenized = tokenizer.apply_chat_template(training_prompt, continue_final_message=True, return_tensors="pt").to(device)
test_out = model.generate(test_tokenized, max_new_tokens=10)
print(tokenizer.batch_decode(test_out, skip_special_tokens=True)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Who to subscribe to on YT for ML?assistant

Subscribe to these popular YouTube channels for Machine Learning (ML)


In [18]:
data = generate_input_output_pair(prompt=[training_prompt], target_responses=[target_response])
data["input_ids"] = data["input_ids"].to(device)
data["labels"] = data["labels"].to(device)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

for _ in range(10):
    out = model(input_ids=data["input_ids"].to(device))
    loss = calculate_loss(out.logits, data["labels"]).mean()

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print("loss: ", loss.item())

loss:  1.1328125
loss:  0.87890625
loss:  0.55859375
loss:  0.32421875
loss:  0.022216796875
loss:  0.0022735595703125
loss:  0.0010223388671875
loss:  0.00070953369140625
loss:  0.0004329681396484375
loss:  0.000278472900390625


In [19]:
test_tokenized = tokenizer.apply_chat_template(training_prompt, continue_final_message=True, return_tensors="pt").to(device)
test_out = model.generate(test_tokenized, max_new_tokens=10)
print(tokenizer.batch_decode(test_out, skip_special_tokens=True)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Who to subscribe to on YT for ML?assistant

Subscribe to Neural Breakdown with AVB
